<a href="https://colab.research.google.com/github/MichaelangeloVelalopoulos/diploma-energy-market/blob/main/notebooks/LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ======================================================================================================================
# MTU (96x15min) LSTM Seq2Seq Residual (LSTM-ONLY) + Leakage-Safe Metrics + CSV Export
#
# Goal:
#   Predict next-day intraday MCP(t) for t=1..96 (15-min MTU resolution).
#
# Modeling idea (Residual learning):
#   We model residual r(t) = MCP(t) - DAM_MCP(t)
#   LSTM predicts r_hat(t) for the next day (96 points).
#   Final prediction: MCP_hat(t) = DAM_MCP(t) + r_hat(t)
#
# Leakage-safe setup:
#   - Time-ordered day split into TRAIN / VAL / TEST (no shuffling)
#   - Scaler fit only on TRAIN sequences
#   - LSTM trained on TRAIN, early-stopped on VAL, evaluated on TEST
#
# Outputs:
#   /content/mtu_preds_LSTM_ONLY.csv
#   /content/metrics_LSTM_ONLY.csv
# ======================================================================================================================

import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd

from dataclasses import dataclass
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras import layers, Model, callbacks

# ---------------------------
# CONFIG
# ---------------------------
@dataclass
class CFG:
    DATA_PATH: str = "/content/Final2026.csv"
    TS_COL: str = "DELIVERY_MTU"
    TARGET_COL: str = "MCP"
    DAM_COL: str = "DAM_MCP"

    # MTU settings
    FREQ_MIN: int = 15
    MTU_PER_DAY: int = 96

    # Optional block feature
    BLOCK_HOURS: int = 4  # 6 blocks/day
    LOOKBACK_DAYS: int = 7

    # Day split (chronological)
    TRAIN_FRAC: float = 0.70
    VAL_FRAC: float   = 0.15
    TEST_FRAC: float  = 0.15

    # Training
    SEED: int = 42
    EPOCHS: int = 30
    BATCH: int = 16
    LR: float = 1e-3
    PATIENCE: int = 6

    # Leakage policy
    INCLUDE_BM_IMBALANCE_PRICE: bool = False

    # Export
    OUT_PRED_CSV: str = "/content/mtu_preds_LSTM_ONLY.csv"
    OUT_METRICS_CSV: str = "/content/metrics_LSTM_ONLY.csv"

cfg = CFG()

np.random.seed(cfg.SEED)
tf.random.set_seed(cfg.SEED)

# ---------------------------
# METRICS
# ---------------------------
def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def safe_r2(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    if np.std(y_true) < 1e-12:
        return np.nan
    return float(r2_score(y_true, y_pred))

def smape(y_true, y_pred, eps=1e-6) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum((np.abs(y_true) + np.abs(y_pred)), eps)
    return float(np.mean(2.0 * np.abs(y_pred - y_true) / denom) * 100.0)

def safe_mape(y_true, y_pred, thresh=10.0, eps=1e-6) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.abs(y_true) >= thresh
    if mask.sum() == 0:
        return np.nan
    denom = np.maximum(np.abs(y_true[mask]), eps)
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / denom)) * 100.0)

def fsi_vs_baseline(y_true, y_pred, y_base, eps=1e-12) -> float:
    rb = rmse(y_true, y_base)
    rm_ = rmse(y_true, y_pred)
    if rb < eps:
        return np.nan
    return float(1.0 - (rm_ / rb))

def metrics_row(name, y_true, y_pred, y_base=None):
    row = {
        "set": name,
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": rmse(y_true, y_pred),
        "R2": safe_r2(y_true, y_pred),
        "sMAPE%": smape(y_true, y_pred),
        "MAPE_safe%": safe_mape(y_true, y_pred, thresh=10.0),
    }
    if y_base is not None:
        row["FSI_vs_DAM"] = fsi_vs_baseline(y_true, y_pred, y_base)
        row["RMSE_DAM_baseline"] = rmse(y_true, y_base)
    return row

# ---------------------------
# LOAD + PREP
# ---------------------------
df = pd.read_csv(cfg.DATA_PATH)
df[cfg.TS_COL] = pd.to_datetime(df[cfg.TS_COL])
df = df.sort_values(cfg.TS_COL).reset_index(drop=True)

if (not cfg.INCLUDE_BM_IMBALANCE_PRICE) and ("BM_IMBALANCE_PRICE" in df.columns):
    df = df.drop(columns=["BM_IMBALANCE_PRICE"])

# time fields
df["day"] = df[cfg.TS_COL].dt.floor("D")
df["minute_of_day"] = df[cfg.TS_COL].dt.hour * 60 + df[cfg.TS_COL].dt.minute
df["mtu_idx"] = (df["minute_of_day"] // cfg.FREQ_MIN).astype(int)  # 0..95
df["block_id"] = (df[cfg.TS_COL].dt.hour // cfg.BLOCK_HOURS).astype(int)  # 0..5

df["dow"] = df[cfg.TS_COL].dt.dayofweek.astype(int)
df["hour"] = df[cfg.TS_COL].dt.hour.astype(int)

# seasonality (ex-ante)
df["sin_hour"] = np.sin(2*np.pi*df["hour"]/24.0)
df["cos_hour"] = np.cos(2*np.pi*df["hour"]/24.0)
df["sin_dow"]  = np.sin(2*np.pi*df["dow"]/7.0)
df["cos_dow"]  = np.cos(2*np.pi*df["dow"]/7.0)

# ---------------------------
# D-1 SAME-MTU alignment for BMmDAMMCP (merge by timestamp)
# ---------------------------
# This guarantees: for timestamp T in day D, BMmDAMMCP_Dm1 comes from timestamp (T - 1 day).
if "BMmDAMMCP" not in df.columns:
    raise ValueError("Missing column BMmDAMMCP in Final2026.csv")

lookup = df[[cfg.TS_COL, "BMmDAMMCP"]].copy()
lookup[cfg.TS_COL] = lookup[cfg.TS_COL] + pd.Timedelta(days=1)   # yesterday's values become available at today's timestamp
lookup = lookup.rename(columns={"BMmDAMMCP": "BMmDAMMCP_Dm1"})
df = df.merge(lookup, on=cfg.TS_COL, how="left")

# Drop first day (no D-1)
df = df.dropna(subset=["BMmDAMMCP_Dm1"]).reset_index(drop=True)

# Keep only full days with 96 MTUs
counts = df.groupby("day")["mtu_idx"].nunique()
full_days = counts[counts == cfg.MTU_PER_DAY].index
df = df[df["day"].isin(full_days)].reset_index(drop=True)
df = df.sort_values(["day", "mtu_idx"]).reset_index(drop=True)

# ---------------------------
# Features (ex-ante, available before delivery)
# ---------------------------
required_cols = [
    cfg.TARGET_COL, cfg.DAM_COL,
    "SystemLoad_DA_forecast", "Wind_DA_forecast", "Solar_DA_forecast",
    "BMmDAMMCP_Dm1",
    "sin_hour", "cos_hour", "sin_dow", "cos_dow",
    "mtu_idx", "block_id"
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in CSV: {missing}")

FEATURES = [
    cfg.DAM_COL,
    "SystemLoad_DA_forecast",
    "Wind_DA_forecast",
    "Solar_DA_forecast",
    "BMmDAMMCP_Dm1",
    "sin_hour", "cos_hour", "sin_dow", "cos_dow",
    "mtu_idx", "block_id"
]
F = len(FEATURES)

# ---------------------------
# Build day tensors
# ---------------------------
days = sorted(df["day"].unique())
day_groups = {d: g for d, g in df.groupby("day", sort=True)}
D = len(days)

print("Total full days:", D)

X_day = np.zeros((D, cfg.MTU_PER_DAY, F), dtype=np.float32)
Y_mcp = np.zeros((D, cfg.MTU_PER_DAY), dtype=np.float32)
Y_dam = np.zeros((D, cfg.MTU_PER_DAY), dtype=np.float32)

for i, d in enumerate(days):
    g = day_groups[d].sort_values("mtu_idx")
    X_day[i] = g[FEATURES].values.astype(np.float32)
    Y_mcp[i] = g[cfg.TARGET_COL].values.astype(np.float32)
    Y_dam[i] = g[cfg.DAM_COL].values.astype(np.float32)

# residual target
Y_resid = (Y_mcp - Y_dam).astype(np.float32)

# ---------------------------
# Build samples: use LOOKBACK_DAYS history -> predict next day residual curve (96 points)
# ---------------------------
first = cfg.LOOKBACK_DAYS
N = D - first
if N < 20:
    raise ValueError(f"Not enough usable days after lookback: N={N}")

usable_days = np.array(days[first:])

n_train = int(N * cfg.TRAIN_FRAC)
n_val   = int(N * (cfg.TRAIN_FRAC + cfg.VAL_FRAC))

idx_train = np.arange(0, n_train)
idx_val   = np.arange(n_train, n_val)
idx_test  = np.arange(n_val, N)

train_days = usable_days[idx_train]
val_days   = usable_days[idx_val]
test_days  = usable_days[idx_test]

print("Usable after lookback:", N)
print("Split days:", len(train_days), len(val_days), len(test_days))

L = cfg.LOOKBACK_DAYS * cfg.MTU_PER_DAY  # timesteps in input sequence

X_seq = np.zeros((N, L, F), dtype=np.float32)
Y_next_resid = np.zeros((N, cfg.MTU_PER_DAY), dtype=np.float32)
Y_next_mcp   = np.zeros((N, cfg.MTU_PER_DAY), dtype=np.float32)
Y_next_dam   = np.zeros((N, cfg.MTU_PER_DAY), dtype=np.float32)

for i in range(N):
    tgt = first + i
    hist = slice(tgt - cfg.LOOKBACK_DAYS, tgt)

    X_seq[i] = X_day[hist].reshape(L, F)     # concatenated 7 days -> (7*96, F)
    Y_next_resid[i] = Y_resid[tgt]           # next day residual curve (96,)
    Y_next_mcp[i]   = Y_mcp[tgt]             # next day MCP curve (96,)
    Y_next_dam[i]   = Y_dam[tgt]             # next day DAM curve (96,)

# ---------------------------
# Scaling (fit ONLY on train sequences)
# ---------------------------
scaler = StandardScaler()
scaler.fit(X_seq[idx_train].reshape(-1, F))

def scale_seq(X):
    Xf = X.reshape(-1, F)
    Xf = scaler.transform(Xf)
    return Xf.reshape(X.shape).astype(np.float32)

X_tr = scale_seq(X_seq[idx_train])
X_va = scale_seq(X_seq[idx_val])
X_te = scale_seq(X_seq[idx_test])

y_tr = Y_next_resid[idx_train]
y_va = Y_next_resid[idx_val]
y_te = Y_next_resid[idx_test]

mcp_va_true = Y_next_mcp[idx_val]
mcp_te_true = Y_next_mcp[idx_test]
dam_va = Y_next_dam[idx_val]
dam_te = Y_next_dam[idx_test]

# ---------------------------
# LSTM Seq2Seq residual model
# ---------------------------
def build_seq2seq_resid(input_len, feat_dim, out_len, lr) -> Model:
    """
    Encoder-Decoder LSTM:
      - Encoder reads the past (LOOKBACK_DAYS*96 timesteps) -> context vector
      - Decoder generates the next-day residual curve (96 timesteps)
    """
    inp = layers.Input(shape=(input_len, feat_dim), name="X_seq")

    # Encoder compresses history -> latent state
    enc = layers.LSTM(64, return_sequences=False, name="enc")(inp)
    enc = layers.Dropout(0.20)(enc)

    # Repeat context 96 times, decode into a sequence
    rep = layers.RepeatVector(out_len, name="repeat_ctx")(enc)
    dec = layers.LSTM(64, return_sequences=True, name="dec")(rep)
    dec = layers.Dropout(0.10)(dec)

    # Predict residual at each MTU
    out = layers.TimeDistributed(layers.Dense(1), name="resid_hat")(dec)   # (B,96,1)
    out = layers.Reshape((out_len,), name="resid_hat_flat")(out)           # (B,96)

    model = Model(inp, out, name="LSTM_Seq2Seq_Residual")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr, clipnorm=1.0),
        loss=tf.keras.losses.Huber(delta=5.0),  # robust to spikes
    )
    return model

tf.keras.backend.clear_session()
lstm = build_seq2seq_resid(L, F, cfg.MTU_PER_DAY, cfg.LR)

es = callbacks.EarlyStopping(monitor="val_loss", patience=cfg.PATIENCE, restore_best_weights=True)

lstm.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=cfg.EPOCHS,
    batch_size=cfg.BATCH,
    callbacks=[es],
    verbose=1
)

# Predict residual curves
pred_resid_va = lstm.predict(X_va, verbose=0).astype(np.float32)
pred_resid_te = lstm.predict(X_te, verbose=0).astype(np.float32)

# Convert to MCP by adding DAM baseline
pred_lstm_va = (dam_va + pred_resid_va).astype(np.float32)
pred_lstm_te = (dam_te + pred_resid_te).astype(np.float32)

# ---------------------------
# METRICS (MTU-level) — flatten all MTUs
# ---------------------------
def flatten_days(curves_2d):
    return curves_2d.reshape(-1)

y_va_flat = flatten_days(mcp_va_true)
y_te_flat = flatten_days(mcp_te_true)

p_va_flat = flatten_days(pred_lstm_va)
p_te_flat = flatten_days(pred_lstm_te)

dam_va_flat = flatten_days(dam_va)
dam_te_flat = flatten_days(dam_te)

metrics = []
metrics.append(metrics_row("VAL_LSTM_MTU",  y_va_flat, p_va_flat, dam_va_flat))
metrics.append(metrics_row("TEST_LSTM_MTU", y_te_flat, p_te_flat, dam_te_flat))

metrics_df = pd.DataFrame(metrics)

print("\n=== METRICS (MTU-level, LSTM-ONLY) ===")
print(metrics_df.sort_values("set").to_string(index=False))

# ---------------------------
# EXPORT CSV (VAL + TEST MTU rows)
# ---------------------------
def build_rows(split_name, split_days, true_curves, dam_curves, pred_curves):
    rows = []
    for i, d in enumerate(split_days):
        g = day_groups[pd.Timestamp(d)].sort_values("mtu_idx").copy()
        rows.append(pd.DataFrame({
            "split": split_name,
            "day": pd.to_datetime(d).date().isoformat(),
            "timestamp": g[cfg.TS_COL].values,
            "mtu_idx": g["mtu_idx"].values.astype(int),
            "block_id": g["block_id"].values.astype(int),
            "true_MCP": true_curves[i].astype(float),
            "DAM_MCP": dam_curves[i].astype(float),
            "pred_LSTM": pred_curves[i].astype(float),
            "err_LSTM": (pred_curves[i] - true_curves[i]).astype(float),
            "abs_err_LSTM": np.abs(pred_curves[i] - true_curves[i]).astype(float),
        }))
    return pd.concat(rows, axis=0).reset_index(drop=True)

preds_val_df = build_rows("VAL",  val_days,  mcp_va_true, dam_va, pred_lstm_va)
preds_te_df  = build_rows("TEST", test_days, mcp_te_true, dam_te, pred_lstm_te)

preds_df = pd.concat([preds_val_df, preds_te_df], axis=0).reset_index(drop=True)

preds_df.to_csv(cfg.OUT_PRED_CSV, index=False)
metrics_df.to_csv(cfg.OUT_METRICS_CSV, index=False)

print("\nSaved:")
print(" - Predictions:", cfg.OUT_PRED_CSV)
print(" - Metrics    :", cfg.OUT_METRICS_CSV)

# ---------------------------
# SANITY PRINT (first TEST day)
# ---------------------------
if len(test_days) > 0:
    j = 0
    d0 = test_days[0]
    print("\n" + "-" * 120)
    print("SANITY CHECK — First TEST day:", pd.to_datetime(d0).date().isoformat())
    print("mtu_idx | True | DAM | LSTM")
    print("-" * 120)
    for t in range(0, 96, 8):  # every 2 hours
        print(f"{t:>6} | {mcp_te_true[j,t]:>6.2f} | {dam_te[j,t]:>6.2f} | {pred_lstm_te[j,t]:>6.2f}")
    print("-" * 120)

preds_df.head(12)


Total full days: 102
Usable after lookback: 95
Split days: 66 14 15
Epoch 1/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 7s 379ms/step - loss: 15.2873 - val_loss: 8.5876
Epoch 2/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 162ms/step - loss: 14.7934 - val_loss: 8.6029
Epoch 3/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 158ms/step - loss: 14.7341 - val_loss: 8.5350
Epoch 4/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 160ms/step - loss: 14.6377 - val_loss: 8.5392
Epoch 5/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 158ms/step - loss: 14.6673 - val_loss: 8.5974
Epoch 6/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 159ms/step - loss: 14.6565 - val_loss: 8.6377
Epoch 7/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 190ms/step - loss: 14.7052 - val_loss: 8.6436
Epoch 8/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 188ms/step - loss: 14.6215 - val_loss: 8.6782
Epoch 9/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 156ms/step - loss: 14.5974 - val_loss: 8.7390

=== METRICS (MTU-level, LSTM-ONLY) ===
          set      MAE     RMSE       R2   sMAPE%  MAPE_safe%  FSI_vs_DAM  RMSE_DAM_baseline
TEST_LSTM_MTU 4.010068 5.486121 0.

,split,day,timestamp,mtu_idx,block_id,true_MCP,DAM_MCP,pred_LSTM,err_LSTM,abs_err_LSTM
0,VAL,2025-12-20,2025-12-20 00:00:00,0,0,104.809998,103.809998,103.588440,-1.221558,1.221558
1,VAL,2025-12-20,2025-12-20 00:15:00,1,0,101.989998,100.989998,100.606842,-1.383156,1.383156
2,VAL,2025-12-20,2025-12-20 00:30:00,2,0,94.599998,95.169998,94.664711,0.064713,0.064713
3,VAL,2025-12-20,2025-12-20 00:45:00,3,0,93.559998,93.820000,93.222382,-0.337616,0.337616
4,VAL,2025-12-20,2025-12-20 01:00:00,4,0,95.760002,101.570000,100.902512,5.142509,5.142509
5,VAL,2025-12-20,2025-12-20 01:15:00,5,0,90.830002,96.769997,96.049637,5.219635,5.219635
6,VAL,2025-12-20,2025-12-20 01:30:00,6,0,90.339996,90.000000,89.239655,-1.100342,1.100342
7,VAL,2025-12-20,2025-12-20 01:45:00,7,0,85.570000,89.540001,88.749481,3.179482,3.179482
8,VAL,2025-12-20,2025-12-20 02:00:00,8,0,86.239998,89.599998,88.786758,2.546761,2.546761
9,VAL,2025-12-20,2025-12-20 02:15:00,9,0,96.260002,94.330002,93.499725,-2.760277,2.760277
